### Figure out GraphSAGE ###

Define bins and bin dictionary

In [ ]:
import numpy as np
bins = [0, .25, .5, .75, 1.0]
#bins = [0, 0.5, 1.0]
bin_dict: dict[float: str] = dict()
for i in range(len(bins)-1):
    bin_name = str(int(np.round(bins[i]*100,0))) + "% -- " + str(int(np.round(bins[i+1]*100,0))) + "%"
    bin_dict[bins[i]] = bin_name
print(bin_dict)

Create visualization routine

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
def visualize(h, data, title = "Scatter Plot"):
    colorlist = ['#e41a1c', '#984ea3', '#377eb8', '#4daf4a', '#ff7f00', '#ffff33', '#a65628']
    z = TSNE(n_components=2).fit_transform(h.detach().cpu().numpy())

    plt.figure(figsize=(10,10))
    plt.xticks([])
    plt.yticks([])
    
    for class_number in range(data.num_classes):
        index_list = extract_nodes_by_class(data.y,class_number)
        #a = [alpha[i] for i in index_list]
        #plt.scatter(z[index_list, 0], z[index_list, 1], s=10, c=colorlist[class_number], alpha = a)
        plt.scatter(z[index_list, 0], z[index_list, 1], s=10, c=colorlist[class_number], alpha = 0.7)

    #plt.scatter(z[:, 0], z[:, 1], s=70, c=color, cmap="Set2")
    _ = plt.legend(bin_dict.values(),bbox_to_anchor=(1, 1), loc='upper left')
    plt.title(title)
    plt.show()

def extract_nodes_by_class(data,class_number):
    index_list = []
    for index, item in enumerate(data):
        if data[index] == class_number:
            index_list.append(index)
    return index_list

Read file directly from pickle file containing pytorch data object

In [ ]:
import pickle

filename = './pickle_data/tensor_type nominal NUM_AGENTS 10 SITE_0_QUALITY 1.0 SITE_1_QUALITY 0.5 QUORUM_THRESHOLD 2.0 UNCERTAIN_NODES_IN_TRAINING False TEST_SIZE 0.3'
# filename = './pickle_data/tensor_type nominal NUM_AGENTS 10 SITE_0_QUALITY 1.0 SITE_1_QUALITY 0.5 QUORUM_THRESHOLD 3.0 UNCERTAIN_NODES_IN_TRAINING False TEST_SIZE 0.3'
# filename = './pickle_data/tensor_type nominal NUM_AGENTS 10 SITE_0_QUALITY 1.0 SITE_1_QUALITY 0.75 QUORUM_THRESHOLD 2.0 UNCERTAIN_NODES_IN_TRAINING False TEST_SIZE 0.3'
# filename = './pickle_data/tensor_type nominal NUM_AGENTS 20 SITE_0_QUALITY 1.0 SITE_1_QUALITY 0.5 QUORUM_THRESHOLD 4.0 UNCERTAIN_NODES_IN_TRAINING False TEST_SIZE 0.3'
# filename = './pickle_data/tensor_type time NUM_AGENTS 10 SITE_0_QUALITY 1.0 SITE_1_QUALITY 0.5 QUORUM_THRESHOLD 2.0 UNCERTAIN_NODES_IN_TRAINING False TEST_SIZE 0.3'
# filename = './pickle_data/tensor_type time NUM_AGENTS 10 SITE_0_QUALITY 1.0 SITE_1_QUALITY 0.5 QUORUM_THRESHOLD 3.0 UNCERTAIN_NODES_IN_TRAINING False TEST_SIZE 0.3'
# filename = './pickle_data/tensor_type time NUM_AGENTS 10 SITE_0_QUALITY 1.0 SITE_1_QUALITY 0.75 QUORUM_THRESHOLD 2.0 UNCERTAIN_NODES_IN_TRAINING False TEST_SIZE 0.3'
# filename = './pickle_data/tensor_type time NUM_AGENTS 20 SITE_0_QUALITY 1.0 SITE_1_QUALITY 0.5 QUORUM_THRESHOLD 4.0 UNCERTAIN_NODES_IN_TRAINING False TEST_SIZE 0.3'

with open(filename, 'rb') as f:
    data = pickle.load(f)
print(data)

---

I'd like to be able to scale the graph neural network classifiers to much larger data sets , to learn embeddings from incomplete graph data, and if possible, run it in real-time while agents are moving around. My first attempt at scaling up will be to use the GraphSAGE algorithm. 

__Learn about the Degree Distribution__

The degree distribution determines how we do neighborhood sampling in GraphSAGE. Let's inspect the degree distribution.

In [ ]:
from collections import Counter
from torch_geometric.utils import to_networkx
import matplotlib.pyplot as plt

G = to_networkx(data)
degree_list = [y for (x,y) in G.degree]
degree_count = Counter(degree_list)
fig, ax = plt.subplots()
ax.set_xlabel('Node degree')
ax.set_ylabel('Number of nodes')
_ = plt.bar(degree_count.keys(),degree_count.values())

ax.set_yscale('log')

The take away from this plot is that the graph has some highly connected hub nodes, so the random neighborhood sampling from GraphSAGE is a good fit.


__Learn about neighbor sampling for mini-batches__

GraphSAGE requires uses mini-batches to enable scaling to very large graphs. The key idea is to use neighbor sampling to form a _batch_, which consists of a subgraph centered around a node. This section explores how mini-batches are created using neighborhood sampling.

See if I can get some sense of what the subgraphs look like when I create some batches. I'll follow the pattern from Maxime's book "Hands-On Graph Neural Networks Using Python"; see pages 130-133.

In [ ]:
from torch_geometric.loader import NeighborLoader

train_loader = NeighborLoader(
    data,
    num_neighbors = [2,2], # three levels deep
    batch_size = 2,
    input_nodes = data.train_mask,
)

Inspect the batches

In [ ]:
for i, batch in enumerate(train_loader):
    print(f"Data batch {i}: {batch}")

Show sample of subgraphs

In [ ]:
from torch_geometric.utils import to_networkx
import networkx as nx

for idx, batch in enumerate(train_loader):
    print(f"Data batch {i}: {batch}")
    H = to_networkx (batch, to_undirected=True)
    print(f"\tsubgraph {idx} has {len(H.nodes)} nodes")
    plt.figure()
    ax = plt.gca()
    ax.set_title(f"Subgraph {idx}", fontsize=24)
    plt.axis('off')
    nx.draw_networkx(H,
                     pos = nx.spring_layout(H, seed=0),
                     #pos = nx.nx_pydot.graphviz_layout(H, prog="neato"),
                     with_labels = False,
                     node_color = batch.y,
                     node_size = 10
    )
    if idx == 2: break
    plt.show()


Each sample produced by neighborhood sampling yields a batch of subgraphs, and each subgraph in a batch is a node plus its neighbors and its neighbors' neighbors. For the example above, I only went two levels deep and only sampled two neighbors per level, so we usually end up with two disconnected subgraphs, each with 1 + 2 + 2*2 nodes corresponding to the node, two of its children, and two of its children's children. Sometimes, the neighborhood sampling resamples the first node when it looks at children's children, so some subgraphs have 6 nodes instead of seven. And some of the subgraphs in a batch happen to share a node, so you get a connected graph.

_Number of batches_

I don't know why the neighborhood loader chooses a certain number of batches. Does it keep adding batches until every node is part of a batch? The original paper doesn't seem to specify. One hypothesis is that we need the same number of nodes in the batches that we have in the original network. Let's start with the most simple way of sampling nodes. The batch size will be 0 and we'll sample no neighbors of the center node.

In [ ]:
train_loader = NeighborLoader(
    data,
    num_neighbors = [0],
    batch_size = 1,
    input_nodes = data.train_mask,
)

print(f"There are {sum(data.train_mask)} nodes in the training mask")
print(f"There are {data.x.shape[0]} nodes in the graph")

number_nodes_in_batches = 0
number_of_batches = 0
for i, batch in enumerate(train_loader):
    number_of_batches += 1
    number_nodes_in_batches += batch.x.shape[0]

print(f"There are {number_nodes_in_batches} nodes in the {number_of_batches} batches")

The number of batches equaled the number of nodes in the training set. What if the batch size goes to 2?

In [ ]:
train_loader = NeighborLoader(
    data,
    num_neighbors = [0],
    batch_size = 2,
    input_nodes = data.train_mask,
)

print(f"There are {sum(data.train_mask)} nodes in the training mask")
print(f"There are {data.x.shape[0]} nodes in the graph")

number_nodes_in_batches = 0
number_of_batches = 0
for i, batch in enumerate(train_loader):
    number_of_batches += 1
    number_nodes_in_batches += batch.x.shape[0]

print(f"There are {number_nodes_in_batches} nodes in the {number_of_batches} batches")

We get half as many batches. 

What if we start to add neighbors?

In [ ]:
train_loader = NeighborLoader(
    data,
    num_neighbors = [2],
    batch_size = 2,
    input_nodes = data.train_mask,
)

print(f"There are {sum(data.train_mask)} nodes in the training mask")
print(f"There are {data.x.shape[0]} nodes in the graph")

number_nodes_in_batches = 0
number_of_batches = 0
for i, batch in enumerate(train_loader):
    number_of_batches += 1
    number_nodes_in_batches += batch.x.shape[0]

print(f"There are {number_nodes_in_batches} nodes in the {number_of_batches} batches")

We still have the same numbr of batches, which means we form a subgraph by sampling the neighbors for each node in the training set. Confirm with two more tests. First, add two levels of neighbor sampling.

In [ ]:
train_loader = NeighborLoader(
    data,
    num_neighbors = [2,2],
    batch_size = 2,
    input_nodes = data.train_mask,
)

print(f"There are {sum(data.train_mask)} nodes in the training mask")
print(f"There are {data.x.shape[0]} nodes in the graph")

number_nodes_in_batches = 0
number_of_batches = 0
for i, batch in enumerate(train_loader):
    number_of_batches += 1
    number_nodes_in_batches += batch.x.shape[0]

print(f"There are {number_nodes_in_batches} nodes in the {number_of_batches} batches")

Then, make the batch size 3. This should yield 618/3 batches.

In [ ]:
train_loader = NeighborLoader(
    data,
    num_neighbors = [2,2],
    batch_size = 3,
    input_nodes = data.train_mask,
)

print(f"There are {sum(data.train_mask)} nodes in the training mask")
print(f"The number of nodes in the training mask divided by 3 is {sum(data.train_mask)/3}")
print(f"There are {data.x.shape[0]} nodes in the graph")

number_nodes_in_batches = 0
number_of_batches = 0
for i, batch in enumerate(train_loader):
    number_of_batches += 1
    number_nodes_in_batches += batch.x.shape[0]

print(f"There are {number_nodes_in_batches} nodes in the {number_of_batches} batches")

So, we get a single subgraph for each node in the training set. This makes sense because we want to use each labeled node to train the network.

---

_Negative sampling_

Since we aren't doing link prediction, we don't need to know which nodes are not connected to which other nodes. Our task is to use small subgraphs to learn the mapping from node to embedding. So, negative sampling does not apply to our task.


---

_New subset of batches per epoch?_

From copilot:

When training a GraphSAGE (Graph Sample and Aggregation) algorithm, the approach for iterating through neighborhood sampled batches can vary based on the specific implementation and training setup. Let’s explore the common practices:

1. Neighbor Sampling:
  - GraphSAGE involves sampling neighbors around each node to create a neighborhood.
  - During training, you sample a batch of nodes (center nodes) and their corresponding neighborhoods (sampled neighbors).
  - The goal is to learn node representations by aggregating information from these neighborhoods.
  
2. Epochs and Batches:
  - An epoch refers to a complete pass through the entire training dataset.
  - A batch is a subset of data used for one update of the model’s parameters.
  - In each epoch, you iterate through multiple batches to update the model.

3. GraphSAGE Training:
  - When training GraphSAGE, you typically iterate through all sampled neighborhoods (batches) in each epoch.
  - For each batch, you compute the aggregation (e.g., mean, LSTM, or other aggregation functions) over the sampled neighbors.
  - The gradients are then backpropagated to update the model parameters.
  - This process repeats for all batches in the dataset.

4. Different Subsets per Epoch:
  - Some variations exist where you use different subsets of neighborhoods per epoch:
    - __Stochastic Gradient Descent (SGD)__: In each epoch, you randomly shuffle the dataset and sample different batches (neighborhoods) for each iteration.
    - __Mini-batch SGD__: You divide the dataset into mini-batches, and each epoch consists of multiple iterations through these mini-batches.
    - __Full-batch__: Rarely used due to memory constraints; it processes the entire dataset in each epoch.

5. Balancing Trade-offs:
  - The choice of batch size, sampling strategy, and number of epochs affects the trade-off between computational efficiency and model performance.
  - Smaller batch sizes allow more frequent updates but may increase noise.
  - Larger batch sizes improve stability but require more memory.

In summary, GraphSAGE typically iterates through all neighborhood sampled batches in each epoch, ensuring that information from different parts of the graph is used for training. However, variations exist based on the specific training setup and optimization techniques.

---

### 2-Level GraphSAGE ###

The following code is adapted from page 134 of _Hands-On Graph Neural Networks Using Python_ by Maxime Labonne.

In [ ]:
import torch
import torch.nn.functional as F
from torch.nn import Linear
from torch_geometric.nn import SAGEConv
from torch_geometric.nn import GCNConv
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score

BATCH_SIZE = 32

class GraphSAGE_3_layer(torch.nn.Module):
    def __init__(self, dim_in, dim_h, dim_out, drop_out, learning_rate, weighting_method):
        super().__init__()
        self.drop_out = drop_out
        self.learning_rate = learning_rate
        self.weighting_method = weighting_method
        #self.hidden_layer_1 = SAGEConv(dim_in, dim_h*4)
        #self.hidden_layer_1 = SAGEConv(dim_h*4, dim_h*2)
        #self.hidden_layer_1 = SAGEConv(dim_h*2, dim_h)
        self.hidden_layer_1 = GCNConv(dim_in, dim_h*4)
        self.hidden_layer_2 = GCNConv(dim_h*4, dim_h*2)
        self.hidden_layer_3 = GCNConv(dim_h*2, dim_h)
        self.output = Linear(dim_h, dim_out)

    def forward(self, x, edge_index):
        # Don't include dropout since the mini batch process already drops out a lot of information.
        #h = F.dropout(x, p=self.drop_out, training = self.training)
        h = self.hidden_layer_1(x, edge_index)
        h = F.elu(h)
        #h = F.dropout(h, p=self.drop_out, training = self.training)
        h = self.hidden_layer_2(h, edge_index)
        h = F.elu(h)
        #h = F.dropout(h, p=self.drop_out, training = self.training)
        h = self.hidden_layer_3(h, edge_index)
        h = F.elu(h)
        embedding = h
        out = self.output(h)
        #out = F.log_softmax(out, dim=1)
        return out, embedding # both the learned class and the embedding are returned.
    def fit(self, data, epochs):
        # Compute class weights for weighted loss function
        y = data.y.detach()
        class_weight = compute_class_weight(class_weight="balanced", classes=np.unique(y), y=y.numpy())
        class_weight = torch.tensor(class_weight, dtype=torch.float32)
        
        # Select optimizer and loss criterion
        optimizer = torch.optim.Adam(self.parameters(), lr=self.learning_rate)#, weight_decay=0.01)
        criterion = torch.nn.CrossEntropyLoss(weight=class_weight, reduction='mean')
        
        # Select data loader
        train_loader = NeighborLoader(
            data,
            num_neighbors = [2,2,1],
            batch_size = BATCH_SIZE,
            input_nodes = data.train_mask,
)

        self.train()
        for epoch in range(epochs+1):
            total_loss = 0
            for batch in train_loader:
                optimizer.zero_grad()
                out, _ = self(batch.x, batch.edge_index)
                loss = criterion(out[batch.train_mask], batch.y[batch.train_mask])
                acc = self.__get_accuracy__(out[batch.train_mask].argmax(dim=1), batch.y[batch.train_mask])
                loss.backward()
                optimizer.step()
                total_loss += loss.item() * batch.batch_size
            total_loss / len(train_loader.dataset)
                # Every now and then print out progress reports
            if epoch % 100 == 0:
                out, _ = self(data.x, data.edge_index)
                test_loss = criterion(out[data.test_mask], data.y[data.test_mask])
                test_acc = self.__get_accuracy__(out[data.test_mask].argmax(dim=1), data.y[data.test_mask])
                f1 = self.__get_f1_score__(data, out)
                print(f"Epoch {epoch:>3} | Train Loss: {loss:.3f} | Train Acc: {acc*100:>5.2f}% | Test Loss {test_loss: .2f} | Test Acc: {test_acc*100:.2f}% | f1 {f1: .3f}")
        return f1
    
    def get_name(self):
        return "3 layer Batched GCN"

    def __get_f1_score__(self, data, out) -> float:
        if self.weighting_method == 'Macro':
            f1 = f1_score(data.y[data.test_mask], out.argmax(dim=1)[data.test_mask], average='macro')
        else:
            f1 = f1_score(data.y[data.test_mask], out.argmax(dim=1)[data.test_mask], average='weighted')
        return f1
    
    def __get_accuracy__(self, y_pred, y_true) -> float:
        return torch.sum(y_pred==y_true) / len(y_true)

    @torch.no_grad()
    def test(self, data):
        self.eval()
        out, _ = self(data.x, data.edge_index)
        acc = self.__get_accuracy__(out.argmax(dim=1)[data.test_mask], data.y[data.test_mask])
        return acc

Train

In [ ]:
LOWER_CONFIDENCE = 5
EMBEDDING_DIMENSION = 8
DROP_OUT = 0.2
MAX_DEPTH = 5
NUM_EPOCHS = 800
HEADS = 8
F1_WEIGHTING_METHOD = 'Weighted' #"Macro"
VISUALIZE_EMBEDDING = False #True
LEARNING_RATE = 0.001


model = GraphSAGE_3_layer(dim_in = data.num_features, 
                            dim_h = EMBEDDING_DIMENSION, 
                            dim_out = data.num_classes,
                            drop_out=DROP_OUT, 
                            learning_rate=LEARNING_RATE, 
                            weighting_method=F1_WEIGHTING_METHOD)

print(model)

Read graph

In [ ]:
import pickle
filename = './pickle_data/tensor_type nominal NUM_AGENTS 10 SITE_0_QUALITY 1.0 SITE_1_QUALITY 0.5 QUORUM_THRESHOLD 2.0 UNCERTAIN_NODES_IN_TRAINING False TEST_SIZE 0.3'
file_list = [filename]
filename = './pickle_data/tensor_type nominal NUM_AGENTS 10 SITE_0_QUALITY 1.0 SITE_1_QUALITY 0.5 QUORUM_THRESHOLD 3.0 UNCERTAIN_NODES_IN_TRAINING False TEST_SIZE 0.3'
file_list.append(filename)
filename = './pickle_data/tensor_type nominal NUM_AGENTS 10 SITE_0_QUALITY 1.0 SITE_1_QUALITY 0.75 QUORUM_THRESHOLD 2.0 UNCERTAIN_NODES_IN_TRAINING False TEST_SIZE 0.3'
file_list.append(filename)
filename = './pickle_data/tensor_type nominal NUM_AGENTS 20 SITE_0_QUALITY 1.0 SITE_1_QUALITY 0.5 QUORUM_THRESHOLD 4.0 UNCERTAIN_NODES_IN_TRAINING False TEST_SIZE 0.3'
file_list.append(filename)
filename = './pickle_data/tensor_type time NUM_AGENTS 10 SITE_0_QUALITY 1.0 SITE_1_QUALITY 0.5 QUORUM_THRESHOLD 2.0 UNCERTAIN_NODES_IN_TRAINING False TEST_SIZE 0.3'
file_list.append(filename)
filename = './pickle_data/tensor_type time NUM_AGENTS 10 SITE_0_QUALITY 1.0 SITE_1_QUALITY 0.5 QUORUM_THRESHOLD 3.0 UNCERTAIN_NODES_IN_TRAINING False TEST_SIZE 0.3'
file_list.append(filename)
filename = './pickle_data/tensor_type time NUM_AGENTS 10 SITE_0_QUALITY 1.0 SITE_1_QUALITY 0.75 QUORUM_THRESHOLD 2.0 UNCERTAIN_NODES_IN_TRAINING False TEST_SIZE 0.3'
file_list.append(filename)
# filename = './pickle_data/tensor_type time NUM_AGENTS 20 SITE_0_QUALITY 1.0 SITE_1_QUALITY 0.5 QUORUM_THRESHOLD 4.0 UNCERTAIN_NODES_IN_TRAINING False TEST_SIZE 0.3'

with open(filename, 'rb') as f:
    data = pickle.load(f)
# print(data)

Train and inspect results

In [ ]:
from utilities.batch_utilities import BatchUtilities
import numpy as np


for filename in file_list:
    #print(f"Processing {filename}")
    conditions = filename.split()
    with open(filename, 'rb') as f:
        data = pickle.load(f)
    if "batch_utilities" in globals(): del batch_utilities
    batch_utilities = BatchUtilities(data)
    for drop_out in [0.6]: #[0.6, 0.2]:
        for embedding_dimension in [4]: #[8, 4, 2]:
            results: dict[str, list[str, float, str, float]] = dict()
            if "model" in globals(): del model
            model = GraphSAGE_3_layer(dim_in = data.num_features, 
                            dim_h = EMBEDDING_DIMENSION, 
                            dim_out = data.num_classes,
                            drop_out=DROP_OUT, 
                            learning_rate=LEARNING_RATE, 
                            weighting_method=F1_WEIGHTING_METHOD)
            # print(f"model is\n{model}")
            name = model.get_name()
            result_list = []
            result_list.append('GCN f1')
            result_list.append(model.fit(data, epochs = NUM_EPOCHS))
            model.eval()
            out, embedding = model(data.x, data.edge_index)

            ## Apply classifier to embedding
            X = embedding.detach().cpu().numpy()
            if VISUALIZE_EMBEDDING: 
                title = name + "-based Embedding"
                batch_utilities.visualize(X, title = title)
            print("Training Classifier")
            batch_utilities.train_classifier(X, max_depth=MAX_DEPTH)
            title = "Confusion matrix for " + name + " Embedding" 
            batch_utilities.show_confusion_matrix(X,title)
            result_list.append('Classifier f1')
            result_list.append(batch_utilities.get_f1_score(X, weighting_method=F1_WEIGHTING_METHOD))
            batch_utilities.train_on_features(max_depth=MAX_DEPTH)
            result_list.append('Cls f1 feature')
            result_list.append(batch_utilities.get_f1_score_on_features(weighting_method=F1_WEIGHTING_METHOD))
            results[name] = result_list

            print(f"***************************")
            print(f"* Tensor type = {conditions[1]} ")
            print(f"*        {conditions[3]} agents        *")
            print(f"*  Q(s0,s1) = ({conditions[5]}, {conditions[7]})  *")
            print(f"*     Threshold = {conditions[9]}     *")
            print(f"*     Test size = {conditions[13]}     *")
            print(f'* Embedding dimension = {embedding_dimension} *')
            print(f"*     Batch size = {BATCH_SIZE}     *")
            for gat_name in results.keys():
                print(f"*  {gat_name}    *")
                print(f"*     {results[gat_name][0]} = {np.round(results[gat_name][1],3)}      *")
                print(f"*  {results[gat_name][2]} = {np.round(results[gat_name][3],3)}  *")
                print(f"* {results[gat_name][4]} = {np.round(results[gat_name][5],3)}  *")
            print(f"***************************")
            print("\n")

Repeat but using files with only 30% training data

In [ ]:
filename = './pickle_data/tensor_type nominal NUM_AGENTS 10 SITE_0_QUALITY 1.0 SITE_1_QUALITY 0.5 QUORUM_THRESHOLD 2.0 UNCERTAIN_NODES_IN_TRAINING False TEST_SIZE 0.7'
file_list = [filename]
filename = './pickle_data/tensor_type nominal NUM_AGENTS 10 SITE_0_QUALITY 1.0 SITE_1_QUALITY 0.5 QUORUM_THRESHOLD 3.0 UNCERTAIN_NODES_IN_TRAINING False TEST_SIZE 0.7'
file_list.append(filename)
filename = './pickle_data/tensor_type nominal NUM_AGENTS 10 SITE_0_QUALITY 1.0 SITE_1_QUALITY 0.75 QUORUM_THRESHOLD 2.0 UNCERTAIN_NODES_IN_TRAINING False TEST_SIZE 0.7'
file_list.append(filename)
filename = './pickle_data/tensor_type nominal NUM_AGENTS 20 SITE_0_QUALITY 1.0 SITE_1_QUALITY 0.5 QUORUM_THRESHOLD 4.0 UNCERTAIN_NODES_IN_TRAINING False TEST_SIZE 0.7'
file_list.append(filename)
filename = './pickle_data/tensor_type time NUM_AGENTS 10 SITE_0_QUALITY 1.0 SITE_1_QUALITY 0.5 QUORUM_THRESHOLD 2.0 UNCERTAIN_NODES_IN_TRAINING False TEST_SIZE 0.7'
file_list.append(filename)
filename = './pickle_data/tensor_type time NUM_AGENTS 10 SITE_0_QUALITY 1.0 SITE_1_QUALITY 0.5 QUORUM_THRESHOLD 3.0 UNCERTAIN_NODES_IN_TRAINING False TEST_SIZE 0.7'
file_list.append(filename)
filename = './pickle_data/tensor_type time NUM_AGENTS 10 SITE_0_QUALITY 1.0 SITE_1_QUALITY 0.75 QUORUM_THRESHOLD 2.0 UNCERTAIN_NODES_IN_TRAINING False TEST_SIZE 0.7'
file_list.append(filename)
# filename = './pickle_data/tensor_type time NUM_AGENTS 20 SITE_0_QUALITY 1.0 SITE_1_QUALITY 0.5 QUORUM_THRESHOLD 4.0 UNCERTAIN_NODES_IN_TRAINING False TEST_SIZE 0.7'


And now run the batch-based learners

In [ ]:
for filename in file_list:
    #print(f"Processing {filename}")
    conditions = filename.split()
    with open(filename, 'rb') as f:
        data = pickle.load(f)
    if "batch_utilities" in globals(): del batch_utilities
    batch_utilities = BatchUtilities(data)
    for drop_out in [0.6]: #[0.6, 0.2]:
        for embedding_dimension in [4]: #[8, 4, 2]:
            results: dict[str, list[str, float, str, float]] = dict()
            if "model" in globals(): del model
            model = GraphSAGE_3_layer(dim_in = data.num_features, 
                            dim_h = EMBEDDING_DIMENSION, 
                            dim_out = data.num_classes,
                            drop_out=DROP_OUT, 
                            learning_rate=LEARNING_RATE, 
                            weighting_method=F1_WEIGHTING_METHOD)
            # print(f"model is\n{model}")
            name = model.get_name()
            result_list = []
            result_list.append('GCN f1')
            result_list.append(model.fit(data, epochs = NUM_EPOCHS))
            model.eval()
            out, embedding = model(data.x, data.edge_index)

                        ## Apply classifier to embedding
            X = embedding.detach().cpu().numpy()
            if VISUALIZE_EMBEDDING: 
                title = name + "-based Embedding"
                batch_utilities.visualize(X, title = title)
            print("Training Classifier")
            batch_utilities.train_classifier(X, max_depth=MAX_DEPTH)
            title = "Confusion matrix for " + name + " Embedding" 
            batch_utilities.show_confusion_matrix(X,title)
            result_list.append('Classifier f1')
            result_list.append(batch_utilities.get_f1_score(X, weighting_method=F1_WEIGHTING_METHOD))
            batch_utilities.train_on_features(max_depth=MAX_DEPTH)
            result_list.append('Cls f1 feature')
            result_list.append(batch_utilities.get_f1_score_on_features(weighting_method=F1_WEIGHTING_METHOD))
            results[name] = result_list

            print(f"***************************")
            print(f"* Tensor type = {conditions[1]} ")
            print(f"*        {conditions[3]} agents        *")
            print(f"*  Q(s0,s1) = ({conditions[5]}, {conditions[7]})  *")
            print(f"*     Threshold = {conditions[9]}     *")
            print(f"*     Test size = {conditions[13]}     *")
            print(f'* Embedding dimension = {embedding_dimension} *')
            print(f"*     Batch size = {BATCH_SIZE}     *")
            for gat_name in results.keys():
                print(f"*  {gat_name}    *")
                print(f"*     {results[gat_name][0]} = {np.round(results[gat_name][1],3)}      *")
                print(f"*  {results[gat_name][2]} = {np.round(results[gat_name][3],3)}  *")
                print(f"* {results[gat_name][4]} = {np.round(results[gat_name][5],3)}  *")
            print(f"***************************")
            print("\n")

---

# GOTCHA 

In the instance of the GCN, the actual batch size was hard-wired as 32, not 16. I'll save this as a new notebook and repeat the data gathering with the batch size of 16.